# Import

In [ ]:
from collections import Counter, defaultdict
from math import ceil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

SEED = 42
sns.set_theme(style='whitegrid')
OUT_DIR = Path('artifacts')
DATA_FILE = OUT_DIR / 'flows_clean.parquet'
assert DATA_FILE.exists(), 'Hãy chạy 01_eda_preprocess.ipynb trước'
features = [
    'ip_prot', 'dst_port', 'flow_duration', 'fwd_pkt_cnt', 'bwd_pkt_cnt',
    'fwd_pkt_len_tot', 'bwd_pkt_len_tot', 'pkt_len_mean', 'pkt_len_std',
    'bytes_per_s', 'pkt_per_s', 'iat_mean', 'iat_std', 'flag_SYN', 'flag_ack'
]
df = pd.read_parquet(DATA_FILE, columns=features + ['is_attack'])
df['is_attack'] = df['is_attack'].astype(int)
print('Dữ liệu sạch:', df.shape)
display(df['is_attack'].value_counts().rename(index={0: 'benign', 1: 'attack'}))

In [ ]:
# Chia toàn bộ dữ liệu trước. Test giữ nguyên tỷ lệ lớp gốc; chỉ lấy mẫu cân bằng từ train
# để thời gian thí nghiệm hợp lý. Cùng train/test cho tất cả 6 lần fit.
idx_train, idx_test = train_test_split(
    np.arange(len(df)), test_size=0.20, stratify=df['is_attack'], random_state=SEED
)
train_full = df.iloc[idx_train]
test = df.iloc[idx_test].copy()
train = pd.concat([
    group.sample(n=min(80_000, len(group)), random_state=SEED)
    for _, group in train_full.groupby('is_attack')
]).sample(frac=1, random_state=SEED).copy()
X_train, y_train = train[features], train['is_attack']
X_test, y_test = test[features], test['is_attack']
print('Train dùng để fit:', y_train.value_counts().to_dict())
print('Test giữ tỷ lệ gốc:', y_test.value_counts().to_dict())

## Phân hạng thuộc tính trên train

`mutual_info_classif` đánh giá mức liên quan phi tuyến với nhãn. Xếp hạng này dùng để **giải thích** các thuộc tính đã chọn theo hiểu biết mạng, không dùng nhãn test và không chọn lại thuộc tính theo kết quả test. Giá trị thiếu được điền median của mẫu train chỉ cho phép tính MI; các mô hình có imputer fit riêng trong pipeline.

In [ ]:
mi_sample = train.sample(n=min(20_000, len(train)), random_state=SEED)
mi_X = SimpleImputer(strategy='median').fit_transform(mi_sample[features])
mi = mutual_info_classif(mi_X, mi_sample['is_attack'], random_state=SEED,
                         discrete_features=[c in ['ip_prot', 'dst_port', 'flag_SYN', 'flag_ack'] for c in features])
feature_rank = pd.DataFrame({'feature': features, 'mutual_information': mi}).sort_values('mutual_information', ascending=False)
display(feature_rank)
feature_rank.to_csv(OUT_DIR / 'feature_rank_train.csv', index=False)

## Rời rạc hóa giao dịch trên train

Mỗi luồng thành một giao dịch gồm 8 mục thuộc tính và một mục nhãn. Ngưỡng 33%/66% được học trên `X_train` cho biến liên tục; các giá trị bằng nhau có thể tạo ít hơn ba nhóm. `ip_prot` và `flag_SYN` giữ dạng nhóm rời rạc. Tập test được biến đổi bằng đúng ngưỡng train. Mức hỗ trợ 6% và mẫu khai phá 12.000 luồng giúp các luật đủ phổ biến, đồng thời giữ notebook chạy được trên máy cá nhân. Điều này giới hạn phát hiện các kiểu tấn công rất hiếm.

In [ ]:
basket_num = ['flow_duration', 'fwd_pkt_cnt', 'bwd_pkt_cnt',
              'pkt_len_mean', 'bytes_per_s', 'iat_mean']
basket_cat = ['ip_prot', 'flag_SYN']
basket_cols = basket_num + basket_cat
bounds = {}
for col in basket_num:
    values = pd.to_numeric(X_train[col], errors='coerce').replace([np.inf, -np.inf], np.nan)
    bounds[col] = np.unique(values.dropna().quantile([1/3, 2/3]).to_numpy())

def bin_frame(X):
    out = pd.DataFrame(index=X.index)
    for col in basket_num:
        values = pd.to_numeric(X[col], errors='coerce').to_numpy(dtype='float64')
        group = np.searchsorted(bounds[col], values, side='right')
        out[col] = np.where(np.isnan(values), f'{col}=missing',
                            np.char.add(f'{col}=b', group.astype(str)))
    for col in basket_cat:
        values = pd.to_numeric(X[col], errors='coerce').fillna(-999).astype(int)
        if col == 'flag_SYN':
            out[col] = np.where(values.to_numpy() > 0, f'{col}=positive', f'{col}=zero')
        else:
            out[col] = f'{col}=' + values.astype(str)
    return out

mine_idx = pd.concat([
    g.sample(n=min(6_000, len(g)), random_state=SEED).index.to_series()
    for _, g in train.groupby('is_attack')
]).to_numpy()
X_mine = X_train.loc[mine_idx]
y_mine = y_train.loc[mine_idx]
mine_bins = bin_frame(X_mine)
transactions = [frozenset(row) | {f'class={int(y)}'}
                for row, y in zip(mine_bins.itertuples(index=False, name=None), y_mine)]
N = len(transactions)
MIN_SUPPORT = 0.06
MIN_COUNT = ceil(MIN_SUPPORT * N)
print('Giao dịch khai phá:', N, '| min_support:', MIN_SUPPORT, '| min_count:', MIN_COUNT)

## CHARM rút gọn – tidset và closure

Thuật toán duyệt IT-tree theo DFS và áp dụng bốn quan hệ tidset của CHARM: bằng nhau, tập con, tập cha, không bao hàm. Phép đóng thêm mọi mục có mặt trong tất cả giao dịch tương ứng; các tập đóng trùng được gộp. Bản Python này không có diffset và hash tối ưu. Chỉ xét tiền tố đến 3 mục vì luật đầu ra cần 1–2 điều kiện và 1 nhãn; closure có thể dài hơn 3 mục.

In [ ]:
def make_vertical(rows):
    vertical = defaultdict(set)
    for tid, row in enumerate(rows):
        for item in row:
            vertical[item].add(tid)
    return dict(vertical)

vertical = make_vertical(transactions)

def charm_closed(vertical, min_count, max_prefix_len=3):
    # Mỗi node là cặp (itemset, tidset); sắp từ support thấp lên để tăng cơ hội cắt nhánh
    nodes = [(frozenset([item]), tids.copy()) for item, tids in vertical.items()
             if len(tids) >= min_count]
    closed = {}
    def extend(nodes):
        nodes.sort(key=lambda pair: (len(pair[1]), tuple(sorted(pair[0]))))
        i = 0
        while i < len(nodes):
            xi, ti = nodes[i]
            children = []
            j = i + 1
            while j < len(nodes):
                xj, tj = nodes[j]
                both = ti & tj
                if len(both) < min_count:
                    j += 1
                    continue
                merged = xi | xj
                if ti == tj:                 # CHARM property 1
                    xi = merged
                    children = [(x | xj, t) for x, t in children]
                    nodes.pop(j)
                elif ti < tj:                # CHARM property 2
                    xi = merged
                    children = [(x | xj, t) for x, t in children]
                    j += 1
                elif ti > tj:                # CHARM property 3
                    if len(merged) <= max_prefix_len:
                        children.append((merged, both))
                    nodes.pop(j)
                else:                       # CHARM property 4
                    if len(merged) <= max_prefix_len:
                        children.append((merged, both))
                    j += 1
            # Kiểm tra đóng bằng tidset, đồng thời loại các node không đóng trùng nhau
            closure = frozenset(xi | {item for item, tids in vertical.items() if ti <= tids})
            closed[closure] = len(ti)
            if children:
                extend(children)
            i += 1
    extend(nodes)
    return closed

closed_sets = charm_closed(vertical, MIN_COUNT)
print('Số tập phổ biến đóng (phạm vi tìm kiếm đã nêu):', len(closed_sets))

## FP-Growth – FP-tree và conditional pattern base

FP-tree nén các tiền tố giao dịch. Khi khai phá, mỗi mục tạo một conditional pattern base rồi dựng cây điều kiện. Giới hạn tập mục đến 3 để cùng phạm vi ứng viên luật 1–2 điều kiện + nhãn như CHARM.

In [ ]:
class FPNode:
    def __init__(self, item=None, count=0, parent=None):
        self.item, self.count, self.parent = item, count, parent
        self.children = {}

def build_fp_tree(weighted_rows, min_count):
    frequency = Counter()
    for row, weight in weighted_rows:
        for item in row:
            frequency[item] += weight
    frequent = {item: n for item, n in frequency.items() if n >= min_count}
    root = FPNode()
    header = defaultdict(list)
    for row, weight in weighted_rows:
        ordered = sorted((item for item in row if item in frequent),
                         key=lambda item: (-frequent[item], item))
        node = root
        for item in ordered:
            if item not in node.children:
                node.children[item] = FPNode(item, 0, node)
                header[item].append(node.children[item])
            node = node.children[item]
            node.count += weight
    return root, header, frequent

def fp_growth(rows, min_count, max_len=3):
    output = {}
    def mine(weighted_rows, prefix):
        _, header, frequency = build_fp_tree(weighted_rows, min_count)
        for item in sorted(frequency, key=lambda x: (frequency[x], x)):
            pattern = prefix | {item}
            output[frozenset(pattern)] = frequency[item]
            if len(pattern) >= max_len:
                continue
            conditional = []
            for node in header[item]:
                path, parent = [], node.parent
                while parent and parent.item is not None:
                    path.append(parent.item)
                    parent = parent.parent
                if path:
                    conditional.append((frozenset(path), node.count))
            if conditional:
                mine(conditional, pattern)
    mine([(row, 1) for row in rows], frozenset())
    return output

frequent_sets = fp_growth(transactions, MIN_COUNT)
print('Số tập phổ biến FP-Growth:', len(frequent_sets))

## Tạo và so sánh 3 luật mỗi phương pháp

Chỉ xét luật `điều kiện mạng ⇒ attack` để ba biến mới diễn giải được theo bài toán. `coverage` là tỷ lệ giao dịch **thỏa tiền đề**; `support` là tỷ lệ thỏa cả tiền đề và kết luận. Độ phủ chung là hợp của ba tiền đề, không cộng ba tỷ lệ riêng. Luật chọn trên **train mining sample** theo đóng góp phủ thêm, confidence và lift. Bảng đánh giá tập test dùng đúng luật đã cố định.

In [ ]:
def matching_tids(items, vertical):
    result = None
    for item in items:
        result = vertical[item].copy() if result is None else result & vertical[item]
    return result if result is not None else set()

def candidate_rules(itemsets, vertical, n, min_conf=.55):
    base_attack = len(vertical['class=1']) / n
    rows = []
    for pattern, support_count in itemsets.items():
        if 'class=1' not in pattern or 'class=0' in pattern:
            continue
        antecedent = frozenset(pattern - {'class=1'})
        if not 1 <= len(antecedent) <= 2:
            continue
        tids = matching_tids(antecedent, vertical)
        coverage = len(tids) / n
        confidence = support_count / len(tids) if tids else 0
        lift = confidence / base_attack
        if confidence >= min_conf and coverage >= .03 and lift > 1:
            rows.append({'antecedent': antecedent, 'support': support_count / n,
                         'coverage': coverage, 'confidence': confidence, 'lift': lift,
                         'tids': tids})
    # Một tiền đề có thể xuất hiện qua nhiều tập đóng/đường đi; giữ bản có support đúng
    unique = {row['antecedent']: row for row in rows}
    return list(unique.values())

def select_three(candidates, n):
    selected, covered = [], set()
    remaining = candidates.copy()
    while remaining and len(selected) < 3:
        best = max(remaining, key=lambda r: (
            len(r['tids'] - covered) / n * r['confidence'] * r['lift'],
            r['confidence'], r['coverage']))
        selected.append(best)
        covered |= best['tids']
        remaining = [r for r in remaining if r['antecedent'] != best['antecedent']]
    if len(selected) < 3:
        raise ValueError('Không đủ 3 luật; hãy giảm MIN_SUPPORT hoặc min_conf và chạy lại trên train')
    return selected

rules_charm = select_three(candidate_rules(closed_sets, vertical, N), N)
rules_fp = select_three(candidate_rules(frequent_sets, vertical, N), N)

def rule_table(rules, method):
    return pd.DataFrame([{
        'method': method, 'rule': f'Rule_{i}',
        'antecedent': ' & '.join(sorted(r['antecedent'])), 'consequent': 'attack',
        'coverage': r['coverage'], 'support': r['support'],
        'confidence': r['confidence'], 'lift': r['lift']
    } for i, r in enumerate(rules, 1)])

rules_table = pd.concat([rule_table(rules_charm, 'CHARM'), rule_table(rules_fp, 'FP-Growth')], ignore_index=True)
display(rules_table.round(4))
rules_table.to_csv(OUT_DIR / 'association_rules.csv', index=False)

def union_coverage(rules, n):
    return len(set().union(*(r['tids'] for r in rules))) / n

coverage_table = pd.DataFrame([
    {'method': 'CHARM', 'rule_1': rules_charm[0]['coverage'],
     'rule_2': rules_charm[1]['coverage'], 'rule_3': rules_charm[2]['coverage'],
     'union_coverage': union_coverage(rules_charm, N)},
    {'method': 'FP-Growth', 'rule_1': rules_fp[0]['coverage'],
     'rule_2': rules_fp[1]['coverage'], 'rule_3': rules_fp[2]['coverage'],
     'union_coverage': union_coverage(rules_fp, N)}
])
display(coverage_table.round(4))
coverage_table.to_csv(OUT_DIR / 'rule_coverage.csv', index=False)

## Tạo D1 = D0 + ba biến luật

Chọn phương pháp có độ phủ hợp lớn hơn; nếu hòa, ưu tiên CHARM. `Rule_i = 1` khi luồng thỏa **tiền đề** của luật, không dùng nhãn kết luận để tạo biến. Ngưỡng rời rạc hóa và luật đều đã học trên train. Tập test chỉ được biến đổi bằng các quy tắc đã cố định.

In [ ]:
chosen_method = coverage_table.sort_values(['union_coverage', 'method'], ascending=[False, True]).iloc[0]['method']
chosen_rules = rules_charm if chosen_method == 'CHARM' else rules_fp
print('Phương pháp chọn:', chosen_method)

def add_rule_features(X, rules):
    bins = bin_frame(X)
    result = X.copy()
    for i, rule in enumerate(rules, 1):
        mask = pd.Series(True, index=X.index)
        for item in rule['antecedent']:
            col = item.split('=', 1)[0]
            mask &= bins[col].eq(item)
        result[f'Rule_{i}'] = mask.astype('int8')
    return result

X_train_d1 = add_rule_features(X_train, chosen_rules)
X_test_d1 = add_rule_features(X_test, chosen_rules)
display(X_train_d1[features + ['Rule_1', 'Rule_2', 'Rule_3']].head())
assert X_train_d1.shape[1] == X_train.shape[1] + 3

In [ ]:
# Độ phủ trên test là kiểm tra khái quát, không dùng để chọn luật/phương pháp
test_bins = bin_frame(X_test)
test_coverage = {}
for method, rules in [('CHARM', rules_charm), ('FP-Growth', rules_fp)]:
    rule_flags = add_rule_features(X_test, rules)[['Rule_1', 'Rule_2', 'Rule_3']]
    test_coverage[method] = rule_flags.any(axis=1).mean()
display(pd.DataFrame({'method': list(test_coverage),
                      'test_union_coverage': list(test_coverage.values())}).round(4))

## Classification – D0 so với D1

Ba mô hình: Logistic Regression, Decision Tree, Random Forest. Mỗi mô hình chạy cùng train/test ở D0 và D1. Imputer, scaler của Logistic Regression chỉ fit trên train. Báo cáo Accuracy, Precision, Recall và F1 của lớp attack; ma trận nhầm lẫn cho từng mô hình ở D1. Vì train đã lấy mẫu cân bằng còn test giữ tỷ lệ gốc, Precision có thể thấp hơn khi áp dụng thực tế.

In [ ]:
def build_models():
    return {
        'Logistic Regression': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                                              LogisticRegression(max_iter=500, random_state=SEED)),
        'Decision Tree': make_pipeline(SimpleImputer(strategy='median'),
                                       DecisionTreeClassifier(max_depth=12, min_samples_leaf=20,
                                                              random_state=SEED)),
        'Random Forest': make_pipeline(SimpleImputer(strategy='median'),
                                       RandomForestClassifier(n_estimators=80, max_depth=16,
                                                              min_samples_leaf=10, n_jobs=-1,
                                                              random_state=SEED))
    }

metrics_rows, confusion_rows = [], []
for dataset, X_tr, X_te in [('D0', X_train, X_test), ('D1', X_train_d1, X_test_d1)]:
    for model_name, model in build_models().items():
        print('Đang fit:', dataset, model_name)
        model.fit(X_tr, y_train)
        pred = model.predict(X_te)
        metrics_rows.append({
            'dataset': dataset, 'model': model_name,
            'accuracy': accuracy_score(y_test, pred),
            'precision_attack': precision_score(y_test, pred, zero_division=0),
            'recall_attack': recall_score(y_test, pred, zero_division=0),
            'f1_attack': f1_score(y_test, pred, zero_division=0)
        })
        if dataset == 'D1':
            confusion_rows.append((model_name, confusion_matrix(y_test, pred, labels=[0, 1])))

results = pd.DataFrame(metrics_rows)
display(results.round(4))
results.to_csv(OUT_DIR / 'classification_results.csv', index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, metric in zip(axes, ['accuracy', 'f1_attack']):
    sns.barplot(data=results, x='model', y=metric, hue='dataset', ax=ax)
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=20)
    ax.set_title(metric)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, matrix) in zip(axes, confusion_rows):
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['benign', 'attack'], yticklabels=['benign', 'attack'])
    ax.set_title(name + ' (D1)')
    ax.set_xlabel('Dự đoán')
    ax.set_ylabel('Thực tế')
plt.tight_layout()
plt.show()

## So sánh và kết luận thực nghiệm

Coverage đo **phạm vi áp dụng tiền đề luật**; Accuracy/F1 đo **dự đoán lớp**. Hai chỉ số giải đáp hai câu hỏi khác nhau. Ba biến luật là hàm rời rạc của thuộc tính gốc nên không thêm thông tin độc lập; mô hình cây có thể không cải thiện, hoặc giảm do cách chia ngưỡng. Đọc dấu `delta_f1`/`delta_accuracy` từ kết quả thực, không sửa luật bằng test để tìm mức tăng.

**Kết quả chạy với seed 42:** độ phủ hợp trên mẫu train: CHARM rút gọn 68,13%, FP-Growth 86,63%; chọn FP-Growth. F1 lớp attack ở D0→D1: Logistic Regression 0,9440→0,9612; Decision Tree 0,9942→0,9940; Random Forest 0,99590→0,99585. Vậy biến luật giúp Logistic Regression trong phép thử này, còn hai mô hình cây giảm rất nhẹ. FP-Growth có độ phủ cao hơn ở **bộ ba luật được chọn**, không chứng minh bản thân thuật toán luôn tốt hơn CHARM: FP-Growth xét mọi tập phổ biến trong phạm vi còn CHARM chỉ xét tập đóng.

**Hạn chế:** train lấy mẫu cân bằng, test chia ngẫu nhiên trong cùng bộ dữ liệu, tấn công hiếm không được phân lớp riêng, và mọi nhãn attack trong bộ này đều thuộc TCP. Các kết quả cao có thể dựa vào đặc điểm của kịch bản thu thập; trước khi triển khai cần đánh giá trên mạng và thời gian khác.

In [ ]:
wide = results.pivot(index='model', columns='dataset', values=['accuracy', 'f1_attack'])
effect = pd.DataFrame({
    'model': wide.index,
    'delta_accuracy_D1_minus_D0': wide[('accuracy', 'D1')].to_numpy() - wide[('accuracy', 'D0')].to_numpy(),
    'delta_f1_D1_minus_D0': wide[('f1_attack', 'D1')].to_numpy() - wide[('f1_attack', 'D0')].to_numpy()
})
display(effect.round(5))
effect.to_csv(OUT_DIR / 'rule_feature_effect.csv', index=False)